[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/freeform/freeform.ipynb)

# FreeForm Elasticity

This example simulates large elastic deformations of a solid Stanford bunny without a volumetric tetrahedralisation by blending a small family of smooth affine motions. Low-discrepancy quadrature points inside the closed triangulated surface integrate the elastic energy and inertia, while generalized eigenmodes of an isotropic stiffness over reproducing kernels select the smoothest scalar weight fields across the body.

Points, displacements, and affine control states are intrinsic spatial directions, whereas spatial gradients and forces are extrinsic covectors read on directions by pairing via `gradient & direction`. Blending four tetrahedral control displacements per weight mode yields the local deformation `Antivector + (state * (gradients & Antivector)).sites.sum()`, whose Neo-Hookean stress combines the metric pullback `local.adjoint().adjugate()` with the cofactor `local.outermorphism(Span)(Vector * unit_volume) / unit_volume` on two-direction joins so flat collapse never divides by a vanishing determinant. Pulling forces back across `&` via `.adjugate()` assembles both the reduced control-space inertia and every sampled force directly onto the controls.

In matrix continuum-mechanics notation, as in Xiang, Modi, and colleagues' [*FreeForm: Reduced-Order Deformable Simulation from Particle-Based Skinning Eigenmodes*](https://research.nvidia.com/labs/sil/projects/freeform/), the deformation gradient is $F = I + \sum_k d_k \nabla w_k^T$, the Neo-Hookean first Piola-Kirchhoff stress writes the cofactor via the matrix inverse transpose $P = \mu F + \kappa(\det F - \alpha)\det(F)F^{-T}$ whenever $\det F \neq 0$, and assembling the reduced mass $B^T M B$ and generalized forces $B^T f$ flattens the spatial and sample axes so each sample mass repeats across a $3 \times 3$ identity block.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[jax,examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax
import numpy as np

from numga import NumpyContext, Orientation, stack
from numga.algebras import VGA3D as ga
from numga.backend.jax import JaxContext
from numga.topology import Topology
from examples.util.animation import save_figure, show_states
from examples.mechanics.freeform import geometry, render, scenarios
from examples.mechanics.freeform.numerics import minimized_state

mv = NumpyContext(ga).multivector
tetrahedron = Topology(4, [[0, 1, 2, 3]])
Scalar = ga.gatype.scalar()
# Points and displacements are antivectors; gradients and forces are covectors, vectors.
Antivector = ga.gatype.antivector()
Vector = ga.gatype.vector()
# A point with an intrinsic one prepended, on which affine functions are linear.
AffinePoint = ga.gatype(ga.subspace.from_grades([0, 2]), orientation=Orientation.intrinsic)
Evaluation = ga.gatype((Scalar, Scalar))                         # Scalar+ <- Scalar+
Gradient = ga.gatype((Vector, Scalar))                           # Vector+ <- Scalar+
Sampling = ga.gatype((Antivector, Antivector))                   # Antivector- <- Antivector-
Deformation = ga.gatype((Antivector, Antivector))                # Antivector- <- Antivector-
Stress = ga.gatype((Vector, Vector))                             # Vector+ <- Vector+
Mass = ga.gatype((Vector, Antivector))                           # Vector+ <- Antivector-
Span = ga.gatype.antibivector()

# The planes across the three axes, the directions along them, and their unit volume join.
planes = stack((mv.x, mv.y, mv.z))                               # [3] Vector+
directions = planes.dual()                                       # [3] Antivector-
unit_volume = directions[0] & directions[1] & directions[2]      # Antiscalar-
one = mv.antipseudoscalar([1])                                   # Antiscalar-

## Volume samples inside a triangulated surface

The closed triangulated surface of the [Stanford bunny](https://graphics.stanford.edu/data/3Dscanrep/) bounds an elastic solid. Low-discrepancy interior points integrate the elastic energy and inertia, a farthest-point subset of centers carries the reproducing kernels, and the surface triangles display the resulting motion.

In [ ]:
surface_path = scenarios.surface_path
height = scenarios.height
n_quadrature = scenarios.n_quadrature
n_centers = scenarios.n_centers

topology, vertices, samples, sample_volumes, centers = geometry.load_solid(
    surface_path, height, n_quadrature, n_centers
)
V, E, F = topology.n_elements
rest_figure = render.surface_figure(vertices, topology)
save_figure(rest_figure, "freeform_rest")
rest_figure

The shaded surface rests upright on `z = 0` at height `2`, framed with room around the ears for the deformation that follows.

## Reproducing kernels and skinning modes

Reproducing kernels correct smooth Gaussians so their weighted sums reproduce affine functions on `one + queries` regardless of how irregularly the quadrature points fall. Pulling the isotropic rest stiffness `(lame_lambda + 4 * lame_mu) * Vector.dual()(gradient)` and the sample volumes back to the kernel centers via `.adjugate()` turns mode selection into a generalized eigenproblem whose lowest modes vary most gently through the solid, starting with the constant mode.

In Kaolin's PyTorch tensor notation, [the affine basis](https://github.com/NVIDIAGameWorks/kaolin/blob/ee8b7eca1e56b1b9fd1509fc406f8d8ccbc9240b/kaolin/physics/simplicits/rkpm.py#L372-L423) concatenates a column of ones onto the coordinate matrix with a zero-padded $3 \times 3$ identity block for its spatial derivative, and [the modal stiffness](https://github.com/NVIDIAGameWorks/kaolin/blob/ee8b7eca1e56b1b9fd1509fc406f8d8ccbc9240b/kaolin/physics/simplicits/rkpm.py#L222-L234) flattens the sample and coordinate axes into a weighted matrix transpose product.

In [ ]:
radius = scenarios.radius
n_modes = scenarios.n_modes
lame_lambda = scenarios.lame_lambda
lame_mu = scenarios.lame_mu
density = scenarios.density

In [ ]:
def kernels(queries: Antivector, centers: Antivector, radius: float) -> tuple[Evaluation, Gradient]:
    """Reproducing kernels map scalar nodal values at `centers` to interpolated values and spatial gradients at `queries`."""
    points = one + centers                                       # [...] AffinePoint-
    difference = queries[..., None] - centers                    # [..., ...] Antivector-
    gaussian = (-difference.scalar_norm_squared() / radius**2).exp()  # [..., ...] Scalar+

    # Correct the Gaussian kernels so their weighted sum reproduces every affine function.
    moment = (gaussian * (points * points.scalar_product(AffinePoint))).sum(axis=-1)  # [...] AffinePoint- <- AffinePoint-
    correction = moment.solve(one + queries)                     # [...] AffinePoint-
    reproduction = points.scalar_product(correction[..., None])  # [..., ...] Scalar+
    weights = gaussian * reproduction                            # [..., ...] Scalar+

    # Differentiating the moment equation accounts for the correction changing with position.
    gaussian_derivative = gaussian * (-2 / radius**2) * (difference.dual() & Antivector)  # [..., ...] Scalar+ <- Antivector-
    moment_change = (points * gaussian_derivative * reproduction).sum(axis=-1)  # [...] AffinePoint- <- Antivector-
    correction_derivative = moment.solve(Antivector - moment_change)  # [...] AffinePoint- <- Antivector-
    derivatives = gaussian_derivative * reproduction + gaussian * points.scalar_product(correction_derivative[..., None])  # [..., ...] Scalar+ <- Antivector-
    # Each weight's gradient is the covector that reads its change along every direction.
    gradients = (planes * derivatives[..., None](directions)).sum(axis=-1)   # [..., ...] Vector+

    # The node axis becomes the input field of the interpolation and gradient maps.
    return (weights * Scalar).field(1), (gradients * Scalar).field(1)


def skinning_modes(evaluation: Evaluation, gradient: Gradient, sample_volumes: np.ndarray,
                   lame_lambda: np.ndarray, lame_mu: np.ndarray,
                   n_modes: int) -> tuple[Scalar, Scalar]:
    """Generalized eigenmodes of the isotropic elastic stiffness against volume mass give the `n_modes` lowest scalar skinning modes (`[n_modes] Scalar+[...]`)."""
    # Assembling over the quadrature sites integrates all pairs of nodal responses together, each pulled back to the
    # nodes by the adjugate: the mass pairs values with sample volumes, and the stiffness pairs gradients with the
    # flux an isotropic material drives.
    values_at_samples = evaluation.field(0)                      # (Scalar+ <- Scalar+)[... <- ...]
    gradients_at_samples = gradient.field(0)                     # (Vector+ <- Scalar+)[... <- ...]
    mass = values_at_samples.adjugate()((sample_volumes * Scalar.dual()(evaluation)).field(0))   # (Antiscalar- <- Scalar+)[... <- ...]
    # Summing the rest elastic costs of three translation directions gives the isotropic flux along each gradient.
    fluxes = sample_volumes * (lame_lambda + 4 * lame_mu) * Vector.dual()(gradient)   # [...] (Antivector- <- Scalar+)[...]
    stiffness = gradients_at_samples.adjugate()(fluxes.field(0))                       # (Antiscalar- <- Scalar+)[... <- ...]
    eigenvalues, mode_fields = stiffness.eigh(mass)              # [...] Scalar+, [...] Scalar+[...]
    return eigenvalues[:n_modes], mode_fields[:n_modes]


evaluation, gradient = kernels(samples, centers, radius)
mode_eigenvalues, modes = skinning_modes(evaluation, gradient, sample_volumes, lame_lambda, lame_mu, n_modes)

## Blending affine motions

An enclosing tetrahedron of four control corners parametrizes an affine motion through its ***simplex gradients***, where the term ***simplex gradient*** here means each corner's opposite outward face patch divided by the tetrahedron's volume join. Multiplying every scalar skinning mode by these four affine weights gives one shared sampling operator for interior quadrature points and surface vertices alike, while pulling `sampling.dual()` back via `.adjugate()` forms the control-space inertia.

In the matrix notation of the [FreeForm evaluation code](https://github.com/nv-tlabs/freeform-eval/blob/a969a62f765c5bed5b23258321afddaf3f171bd8/sim/simplicits_ext.py#L361-L370), affine weights are barycentric coordinates, the reduced inertia is $B^T M B$ with each scalar sample mass Kronecker-expanded across spatial coordinates, and sampled forces return to the controls through $B^T$.

In [ ]:
corners = mv.vector([[-1, -1, 0], [1, -1, 2], [-1, 1, 2], [1, 1, 0]]).dual()   # [4] Antivector-
n_controls = len(corners) * n_modes

In [ ]:
def deformation_basis(queries: Antivector, centers: Antivector, radius: float,
                      modes: Scalar, corners: Antivector) -> tuple[Sampling, Vector]:
    """Blending four affine control corners per scalar mode yields the displacement sampling map and feature gradients at `queries`."""
    evaluation, gradient = kernels(queries, centers, radius)
    weights = evaluation[:, None](modes)                         # [..., ...] Scalar+
    weight_gradients = gradient[:, None](modes)                  # [..., ...] Vector+

    # Each control corner's simplex gradient is its opposite outward face join divided by the tetrahedron's edge join.
    vertices = corners.field()                                   # Antivector-[4]
    edges = (tetrahedron.oriented[3, 1] * (tetrahedron.d[0] * vertices)).batch()[:, 0]   # [3] Antivector-
    sides = tetrahedron.boundary[2, 1] * (tetrahedron.d[0] * vertices)                   # [3] Antivector-[4]
    face_patches = (tetrahedron.boundary[3, 2] * (sides[0] & sides[1])).batch()[:, 0]    # [4] Span-
    affine_gradients = -face_patches / (edges[0] & edges[1] & edges[2])                  # [4] Vector+
    affine_weights = 1 + (affine_gradients & (queries[:, None] - corners))  # [..., 4] Scalar+

    features = weights[..., None] * affine_weights[:, None, :]   # [..., ..., 4] Scalar+
    derivatives = (weight_gradients[..., None] * affine_weights[:, None, :]
                   + weights[..., None] * affine_gradients)      # [..., ..., 4] Vector+
    # A state is the field of the four control displacements of every affine map.
    sampling = (features.reshape(len(queries), -1) * Antivector).field(1)  # [...] (Antivector- <- Antivector-)[1 <- ...]
    gradients = derivatives.reshape(len(queries), -1).field()    # [...] Vector+[...]
    return sampling, gradients


def control_mass(sampling: Sampling, sample_masses: np.ndarray) -> Mass:
    """Pulling each sample's metric displacement covector back through `sampling` assembles the control-space mass map from accelerations to forces."""
    return sampling.field(0).adjugate()((sampling.dual() * sample_masses).field(0))


sampling, gradients = deformation_basis(samples, centers, radius, modes, corners)
surface_sampling, _ = deformation_basis(vertices, centers, radius, modes, corners)
mass = control_mass(sampling, sample_volumes * density)          # (Vector+ <- Antivector-)[n_controls <- n_controls]

## Hyperelastic energy and stress

The local deformation measures how neighbouring points separate while leaving rigid turns free of elastic cost. Its first Piola-Kirchhoff stress maps each feature gradient to a force covector: the stretch contribution passes through the metric as `local.adjoint().adjugate()`, and the volume contribution uses the outermorphism `local.outermorphism(Span)` on two-direction joins `a & b` so the cofactor stays finite even if a trial step flattens part of the body.

In the matrix notation of [Kaolin's Warp material](https://github.com/NVIDIAGameWorks/kaolin/blob/ee8b7eca1e56b1b9fd1509fc406f8d8ccbc9240b/kaolin/physics/materials/neohookean_elastic_material.py#L77-L87), the stretch invariant is $\operatorname{tr}(F^T F)$ and the volume stress term writes the cofactor as $\det(F)F^{-T}$.

In [ ]:
def deformation(state: Antivector, gradients: Vector) -> Deformation:
    """Blending the control displacements `state` with `gradients` forms the local deformation map from reference directions to deformed directions."""
    return Antivector + (state * (gradients & Antivector)).sites.sum()


def cofactor(local: Deformation) -> Stress:
    """The outermorphism of `local` onto two-direction joins carries each reference plane to its deformed image per unit volume, giving the cofactor map on covectors even at flat collapse."""
    # A plane is the join of two directions that span it, measured in units of the reference volume.
    return local.outermorphism(Span)(Vector * unit_volume) / unit_volume      # [...] Vector+ <- Vector+

In [ ]:
def constitutive(local: Deformation, lame_lambda: float, lame_mu: float) -> tuple[Scalar, Stress]:
    """Stable Neo-Hookean elasticity assigns an energy density (`[...] Scalar+`) and first Piola-Kirchhoff stress (`[...] Vector+ <- Vector+`) to `local`."""
    determinant = local.det()                                    # [...] Scalar+
    bulk = lame_lambda + lame_mu
    target = 1 + lame_mu / bulk
    rest_energy = bulk * (1 - target)**2 + 3 * lame_mu
    # The stretch traces the deformed metric against the reference metric across the three reference directions.
    stretch = local.symmetric_reverse_product().contract()       # [...] Scalar+
    energy = (bulk * (determinant - target).squared() + lame_mu * stretch - rest_energy) / 2
    # The stretch acts through the metric while the volume term acts through the cofactor.
    stress = lame_mu * local.adjoint().adjugate() + bulk * (determinant - target) * cofactor(local)   # [...] Vector+ <- Vector+
    return energy, stress


def potential(state: Antivector, sampling: Sampling, gradients: Vector,
              sample_volumes: np.ndarray, lame_lambda: float, lame_mu: float,
              clamp: Scalar, load: Vector) -> tuple[Scalar, Vector]:
    """Integrating Neo-Hookean elasticity, a compliant base clamp, and an external `load` over `sample_volumes` gives the total energy and control-space force field."""
    displacement = sampling(state)                               # [...] Antivector-
    local = deformation(state, gradients)                        # [...] Antivector- <- Antivector-
    density, stress = constitutive(local, lame_lambda, lame_mu)
    density = density + clamp * displacement.scalar_norm_squared() / 2 - (load & displacement)
    energy = (density * sample_volumes).sum(axis=0)              # Scalar+
    elastic = (stress(gradients) * sample_volumes).sum(axis=0)   # Vector+[...]
    # Pulling the clamp restoring covector and external load back via the adjugate collects local forces onto the controls.
    applied = sampling.field(0).adjugate()(
        ((clamp * displacement.dual() - load) * sample_volumes).field()
    )                                                            # Vector+[...]
    return energy, elastic + applied

## Implicit pull and release

A short force pulse pulls one ear sideways while a compliant support holds the base. After release, stored elastic energy drives the recoil and implicit Euler damps the ringing as each step minimizes the elastic, external, and inertial departure energy with L-BFGS whitened by the control-space inertia.

Compiling the incremental objective with `jax.jit` and stepping 200 implicit iterations takes about 10 seconds.

In [ ]:
clamp_depth = scenarios.clamp_depth
clamp_stiffness = scenarios.clamp_stiffness
pull_center = scenarios.pull_center                              # Antivector-
pull_width = scenarios.pull_width
pull_force = scenarios.pull_force                                # Vector+
pull_duration = scenarios.pull_duration
time_step = scenarios.time_step
n_steps = scenarios.n_steps
n_steps_per_frame = scenarios.n_steps_per_frame
n_iterations = scenarios.n_iterations
tolerance = scenarios.tolerance
frame_duration_ms = scenarios.frame_duration_ms
n_frames = n_steps // n_steps_per_frame + 1

In [ ]:
clamp = (-(mv.z & samples).squared() / clamp_depth**2).exp() * clamp_stiffness  # [n_quadrature] Scalar+
profile = (-(samples - pull_center).scalar_norm_squared() / pull_width**2).exp()  # [n_quadrature] Scalar+
# Normalizing over the solid distributes the stated total force across the profile.
load = profile * pull_force / (profile * sample_volumes).sum(axis=0)  # [n_quadrature] Vector+
pulse_strengths = np.sin(np.pi * np.minimum(np.arange(n_steps) * time_step / pull_duration, 1))**2


@jax.jit
def step_objective(trial: Antivector, prediction: Antivector, strength: float) -> tuple[Scalar, Vector]:
    """Incremental elastic, external, and inertial energy and control-space forces at `trial`."""
    energy, force = potential(trial, sampling, gradients, sample_volumes,
                              lame_lambda, lame_mu, clamp, load * strength)
    # Inertia resists departing from the free-flight prediction.
    departure = trial - prediction                               # Antivector-[n_controls]
    inertial_force = mass(departure) / time_step**2              # Vector+[n_controls]
    return energy + (inertial_force & departure).sites.sum() / 2, force + inertial_force


def bunny_trajectory() -> Iterator[Antivector]:
    """Yields the surface vertex positions (`[V] Antivector-`) every `n_steps_per_frame` steps."""
    context = JaxContext(ga)
    state = mv.vector([[0, 0, 0]] * n_controls).dual().field()   # Antivector-[n_controls]
    velocity = state                                             # Antivector-[n_controls]
    yield vertices + surface_sampling(state)
    for step_index, strength in enumerate(pulse_strengths, 1):
        prediction = (state + velocity * time_step).materialize(context)  # Antivector-[n_controls]

        def objective(trial: Antivector) -> tuple[Scalar, Vector]:
            return tuple(value.materialize(mv.context)
                         for value in step_objective(trial.materialize(context), prediction, strength))

        next_state = minimized_state(state, objective, mass / time_step**2,
                                     n_iterations, tolerance)    # Antivector-[n_controls]
        velocity = (next_state - state) / time_step              # Antivector-[n_controls]
        state = next_state                                       # Antivector-[n_controls]
        if step_index % n_steps_per_frame == 0:
            yield vertices + surface_sampling(state)


recorded_states = show_states(
    bunny_trajectory(), render.surface_frames, topology,
    name="freeform_bunny", duration_ms=frame_duration_ms, n_frames=n_frames,
);

The pulled ear bends smoothly toward the foreground during the pulse and rebounds across the body while the clamped base stays anchored on the floor.

In [ ]:
# checks
# The constant mode has vanishing stiffness eigenvalue and constant gradient.
assert (mode_eigenvalues[0].abs() < 1e-8).all()
assert (gradient(modes[0]).scalar_norm_squared().max() < 1e-10).all()

# The ear follows the pull and springs back after release, while the base moves less than a tenth as far.
surfaces = stack(recorded_states)                                # [n_frames, V] Antivector-
displacement = surfaces - surfaces[0]                            # [n_frames, V] Antivector-
pull_work = pull_force & displacement                            # [n_frames, V] Scalar+
distances = displacement.antinorm()                              # [n_frames, V] Scalar+
base_mask = (mv.z & surfaces[0]) < 0.05                          # [V]
assert (pull_work.max() > 0.3).all()
assert (pull_work[-1].max() < 0.05 * pull_work.max()).all()
assert (distances[:, base_mask].max() < 0.1 * distances.max()).all()